# 04 · Interpretability Analysis

**Central thesis contribution.** This notebook provides quantitative evidence for all three research hypotheses.

| Hypothesis | Question |
|---|---|
| **H1** | Do Ridge and Elastic Net yield more stable coefficients than correlation filtering, at competitive forecast accuracy? |
| **H2** | Does correlation filtering discard predictors that carry residual explanatory power (information loss)? |
| **H3** | At which forecast horizons does LSTM significantly outperform the best SARIMAX variant? |

**Structure:**
1. **§1 Coefficient Stability** — 5-fold rolling CV on training data: Coefficient of Variation (CV = std/|mean|) and sign consistency per predictor × strategy
2. **§2 SHAP Analysis** — XGBoost on Stage-1 test residuals; SHAP values quantify which predictors each strategy failed to capture
3. **§3 Hypothesis Tests** — Quantitative answers to H1 and H2; H3 scaffold for LSTM results (notebook 05)

**Inputs:** `data/strategy_*_{train,test}.csv`, `data/sarimax_mape_table.csv`, strategy param JSON files  
**Outputs:** `data/interpretability_coef_stability.csv`, `data/interpretability_shap_summary.csv`, `data/interpretability_h1_summary.csv`, plots

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

def _conda_install(pkg):
    import subprocess, sys
    conda_exe = os.path.join(os.path.dirname(sys.executable), '..', 'bin', 'conda')
    conda_exe = os.path.normpath(conda_exe)
    if not os.path.exists(conda_exe):
        conda_exe = 'conda'
    subprocess.check_call([conda_exe, 'install', '-y', '-q', '-c', 'conda-forge', pkg])

try:
    import shap
    print(f'shap {shap.__version__}')
except ImportError:
    _conda_install('shap')
    import shap
    print(f'shap {shap.__version__} (just installed)')

try:
    import xgboost as xgb
    print(f'xgboost {xgb.__version__}')
except ImportError:
    _conda_install('xgboost')
    import xgboost as xgb
    print(f'xgboost {xgb.__version__} (just installed)')

os.makedirs('plots', exist_ok=True)
print('All imports OK')

In [ ]:
DATA_DIR = 'data'
PLOT_DIR = 'plots'
N_CV_SPLITS = 5

# Load hyperparameters from notebook 02
with open(f'{DATA_DIR}/strategy_ridge_params.json') as f:
    ridge_params = json.load(f)
with open(f'{DATA_DIR}/strategy_elasticnet_params.json') as f:
    enet_params = json.load(f)

RIDGE_ALPHA = ridge_params['lambda']
ENET_L1     = enet_params['l1_ratio']
ENET_ALPHA  = enet_params['alpha']

# Load train/test demand targets
train_df = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['date'])
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv',  parse_dates=['date'])
y_train  = train_df['demand_MW'].values
y_test   = test_df['demand_MW'].values

# Load strategy feature matrices (pre-scaled, from notebook 02)
def load_split(strategy, split):
    df = pd.read_csv(f'{DATA_DIR}/strategy_{strategy}_{split}.csv', parse_dates=['date'])
    return df.drop(columns=['date'])

s1_tr = load_split('filtered',   'train');  s1_te = load_split('filtered',   'test')
s2_tr = load_split('pca',        'train');  s2_te = load_split('pca',        'test')
s3_tr = load_split('ridge',      'train');  s3_te = load_split('ridge',      'test')
s4_tr = load_split('elasticnet', 'train');  s4_te = load_split('elasticnet', 'test')

s1_cols = list(s1_tr.columns)
s3_cols = list(s3_tr.columns)  # all 18 features (Ridge uses all)
s4_cols = list(s4_tr.columns)

# Features removed by each strategy
filtered_out_s1 = sorted(set(s3_cols) - set(s1_cols))
zeroed_by_enet  = enet_params['zeroed_features']

print(f'Train: {len(y_train):,} days ({train_df.date.min().date()} to {train_df.date.max().date()})')
print(f'Test : {len(y_test):,}  days ({test_df.date.min().date()} to {test_df.date.max().date()})')
print(f'Features — S1(Filter): {len(s1_cols)} | S3(Ridge): {len(s3_cols)} | S4(ElasNet): {len(s4_cols)}')
print(f'Filtered out by S1     : {filtered_out_s1}')
print(f'Zeroed by ElasticNet   : {zeroed_by_enet}')
print(f'Ridge alpha={RIDGE_ALPHA} | ElasNet l1_ratio={ENET_L1}, alpha={ENET_ALPHA}')

---
## §1 Coefficient Stability — 5-Fold Rolling Cross-Validation

We use `TimeSeriesSplit(n_splits=5)` on the training data (4281 days). For each fold, we refit the Stage-1 regression for strategies 1, 3, and 4 (OLS, Ridge, ElasticNet) using the fixed hyperparameters tuned in notebook 02. Strategy 2 (PCA) is excluded because its component coefficients are not interpretable in the original feature space.

**Metrics:**
- **CV (Coefficient of Variation):** `std(coef) / |mean(coef)|` across 5 folds — **lower = more stable**
- **Sign consistency:** fraction of folds where the coefficient's sign matches the overall mean sign — **1.0 = always consistent**

> **Expected result (H1):** Ridge and ElasticNet should show lower CV than Filter/OLS for collinear predictors (temp_c, pressure_hpa, global_rad), because L2/L1 regularisation shrinks coefficients toward a stable solution rather than amplifying multicollinearity.

In [ ]:
# 5-fold rolling CV — refit S1/S3/S4 per fold and record coefficients
tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS)

coef_folds = {'Filter': [], 'Ridge': [], 'ElasticNet': []}
X_s1 = s1_tr.values
X_s3 = s3_tr.values
X_s4 = s4_tr.values

for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_s3)):
    y_fold = y_train[tr_idx]

    # Strategy 1: OLS on filtered feature set
    ols = LinearRegression().fit(X_s1[tr_idx], y_fold)
    coef_folds['Filter'].append(dict(zip(s1_cols, ols.coef_)))

    # Strategy 3: Ridge on all features
    ridge = Ridge(alpha=RIDGE_ALPHA).fit(X_s3[tr_idx], y_fold)
    coef_folds['Ridge'].append(dict(zip(s3_cols, ridge.coef_)))

    # Strategy 4: ElasticNet (l1_ratio=1 = Lasso)
    enet = ElasticNet(l1_ratio=ENET_L1, alpha=ENET_ALPHA, max_iter=5000)
    enet.fit(X_s4[tr_idx], y_fold)
    coef_folds['ElasticNet'].append(dict(zip(s4_cols, enet.coef_)))

    n_tr, n_va = len(tr_idx), len(va_idx)
    print(f'  Fold {fold+1}/{N_CV_SPLITS}: train={n_tr:,}, val={n_va:,}')

print('CV complete.')

In [ ]:
def stability_metrics(coef_list, feature_universe):
    '''Compute CV and sign consistency for a list of fold coefficient dicts.'''
    # Build a DataFrame: rows=folds, cols=features (NaN if feature absent in strategy)
    df = pd.DataFrame(
        [{f: d.get(f, np.nan) for f in feature_universe} for d in coef_list]
    )
    mean_c = df.mean()
    std_c  = df.std()
    # CV = std / |mean|, capped at 5 to avoid div-by-zero inflation
    cv = (std_c / mean_c.abs()).clip(upper=5.0)
    # Sign consistency: fraction of folds matching sign of cross-fold mean
    sign_cons = pd.Series({
        feat: float((np.sign(df[feat].dropna()) == np.sign(mean_c[feat])).mean())
        for feat in feature_universe
        if feat in df.columns and mean_c[feat] != 0
    })
    return mean_c, std_c, cv, sign_cons

# All unique features across the three interpretable strategies
all_feats = sorted(set(s1_cols) | set(s3_cols) | set(s4_cols))

mean_s1, std_s1, cv_s1, sign_s1 = stability_metrics(coef_folds['Filter'],     all_feats)
mean_s3, std_s3, cv_s3, sign_s3 = stability_metrics(coef_folds['Ridge'],      all_feats)
mean_s4, std_s4, cv_s4, sign_s4 = stability_metrics(coef_folds['ElasticNet'], all_feats)

# Build heatmap DataFrames (rows=features, cols=strategies)
cv_hm = pd.DataFrame({
    'Filter (OLS)': cv_s1,
    'Ridge':        cv_s3,
    'ElasticNet':   cv_s4,
}).reindex(all_feats)

sign_hm = pd.DataFrame({
    'Filter (OLS)': sign_s1,
    'Ridge':        sign_s3,
    'ElasticNet':   sign_s4,
}).reindex(all_feats)

print('Median CV across features (lower = more stable):')
print(cv_hm.median().round(3))
print('\nMean sign consistency (higher = more consistent):')
print(sign_hm.mean().round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 9))

# --- CV heatmap ---
cv_plot = cv_hm.astype(float)
# Replace NaN (feature not in strategy) with -0.1 for visual distinction
cv_plot_display = cv_plot.fillna(-0.1)

mask_na = cv_plot.isna()
sns.heatmap(
    cv_plot_display,
    ax=axes[0],
    cmap='YlOrRd',
    vmin=0, vmax=1.5,
    annot=cv_plot.round(2).fillna('—'),
    fmt='',
    linewidths=0.5,
    linecolor='#cccccc',
    cbar_kws={'label': 'CV  (std / |mean|)', 'shrink': 0.8},
    mask=mask_na
)
# Shade excluded cells
for i, feat in enumerate(all_feats):
    for j, strat in enumerate(['Filter (OLS)', 'Ridge', 'ElasticNet']):
        if pd.isna(cv_hm.loc[feat, strat]):
            axes[0].add_patch(mpatches.Rectangle((j, i), 1, 1, fill=True,
                              color='#e8e8e8', zorder=2))
            axes[0].text(j + 0.5, i + 0.5, '—', ha='center', va='center',
                         fontsize=9, color='#888888')
axes[0].set_title('Coefficient of Variation\n(lower = more stable; grey = excluded)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Predictor')
axes[0].tick_params(axis='x', rotation=20)

# --- Sign consistency heatmap ---
sign_plot = sign_hm.astype(float)
mask_na_s = sign_plot.isna()
sns.heatmap(
    sign_plot.fillna(0),
    ax=axes[1],
    cmap='RdYlGn',
    vmin=0, vmax=1,
    annot=sign_plot.round(2).fillna('—'),
    fmt='',
    linewidths=0.5,
    linecolor='#cccccc',
    cbar_kws={'label': 'Sign consistency (fraction of folds)', 'shrink': 0.8},
    mask=mask_na_s
)
for i, feat in enumerate(all_feats):
    for j, strat in enumerate(['Filter (OLS)', 'Ridge', 'ElasticNet']):
        if pd.isna(sign_hm.loc[feat, strat]):
            axes[1].add_patch(mpatches.Rectangle((j, i), 1, 1, fill=True,
                              color='#e8e8e8', zorder=2))
            axes[1].text(j + 0.5, i + 0.5, '—', ha='center', va='center',
                         fontsize=9, color='#888888')
axes[1].set_title('Sign Consistency\n(1.0 = same sign in all folds; grey = excluded)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('')
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('Coefficient Stability: 5-Fold Rolling Time-Series Cross-Validation', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/interpretability_coef_stability.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {PLOT_DIR}/interpretability_coef_stability.png')

---
## §2 SHAP Analysis on Stage-1 Test Residuals

**Approach:** For each strategy, we fit the Stage-1 regression on the full training set and compute out-of-sample residuals on the test set:
$$\text{residual}_{i} = y_i^{\text{demand}} - \hat{y}_i^{\text{Stage-1}}$$

We then fit an XGBoost regressor on these residuals using **all 18 original (scaled) features** — including those that may have been excluded by a given strategy. This lets us ask: *which predictors explain what the Stage-1 model missed?*

**Key question for H2:** Do the features removed by correlation filtering (population, sunshine_h) appear as important SHAP contributors in the Filter strategy's residuals? If yes, filtering discarded useful information.

**Note on PCA:** Strategy 2 (PCA) is included for completeness. Its Stage-1 compresses 9 continuous features into 9 principal components, so its residuals reflect information that PCA failed to reconstruct — analogous to reconstruction error.

In [ ]:
# Fit Stage-1 on full training set for each strategy
stage1_models = {
    'Filter':     LinearRegression().fit(s1_tr.values, y_train),
    'PCA':        LinearRegression().fit(s2_tr.values, y_train),
    'Ridge':      Ridge(alpha=RIDGE_ALPHA).fit(s3_tr.values, y_train),
    'ElasticNet': ElasticNet(l1_ratio=ENET_L1, alpha=ENET_ALPHA, max_iter=5000).fit(s4_tr.values, y_train),
}
test_inputs = {
    'Filter':     s1_te.values,
    'PCA':        s2_te.values,
    'Ridge':      s3_te.values,
    'ElasticNet': s4_te.values,
}

# Compute Stage-1 out-of-sample predictions and residuals
stage1_pred   = {}
stage1_resid  = {}
for name, model in stage1_models.items():
    pred = model.predict(test_inputs[name])
    stage1_pred[name]  = pred
    stage1_resid[name] = y_test - pred

# Summary statistics
print(f'{"Strategy":<12}  {"RMSE (MW)":>10}  {"MAE (MW)":>9}  {"R2":>7}')
print('-' * 42)
for name, resid in stage1_resid.items():
    rmse = np.sqrt(np.mean(resid**2))
    mae  = np.mean(np.abs(resid))
    r2   = 1 - np.var(resid) / np.var(y_test)
    print(f'{name:<12}  {rmse:>10.0f}  {mae:>9.0f}  {r2:>7.4f}')

In [ ]:
# SHAP: use ALL original features as XGBoost input (s3_te = Ridge test = all 18 features)
# This allows SHAP to attribute residuals to ANY feature, including filtered/zeroed ones
X_shap = s3_te.copy()  # DataFrame with proper column names

shap_values  = {}
xgb_r2       = {}

strategy_names  = ['Filter', 'PCA', 'Ridge', 'ElasticNet']
strategy_labels = [
    'Strategy 1: Filter (OLS)',
    'Strategy 2: PCA',
    'Strategy 3: Ridge',
    'Strategy 4: ElasticNet',
]

import builtins

def _shap_tree_explainer(xgb_model):
    """Workaround for XGBoost 2.x / SHAP TreeExplainer incompatibility.

    XGBoost 2.x serialises base_score as '[value]' (e.g. '[-1.14E1]') in its
    internal JSON. SHAP calls float() on that string directly and crashes.
    We temporarily replace builtins.float to strip the brackets, then restore it.
    """
    _orig = builtins.float
    def _float(x):
        if isinstance(x, str) and x.startswith('[') and x.endswith(']'):
            return _orig(x[1:-1])
        return _orig(x)
    builtins.float = _float
    try:
        explainer = shap.TreeExplainer(xgb_model.get_booster())
    finally:
        builtins.float = _orig
    return explainer

for name in strategy_names:
    resid = stage1_resid[name]

    xgb_model = xgb.XGBRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbosity=0
    )
    xgb_model.fit(X_shap, resid)

    explainer   = _shap_tree_explainer(xgb_model)
    sv          = explainer.shap_values(X_shap)
    shap_values[name] = sv

    pred_resid = xgb_model.predict(X_shap)
    r2_xgb = 1 - np.var(resid - pred_resid) / np.var(resid)
    xgb_r2[name] = r2_xgb
    print(f'{name:<12}: XGB R2 on residuals = {r2_xgb:.4f}  |  mean|SHAP| = {np.abs(sv).mean():.1f} MW')

In [ ]:
# SHAP bar plots: mean |SHAP| per feature, one subplot per strategy
# Red bars = features filtered out by Strategy 1 (correlation filter)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for ax, name, label in zip(axes, strategy_names, strategy_labels):
    mean_shap = pd.Series(
        np.abs(shap_values[name]).mean(axis=0),
        index=s3_cols
    ).sort_values(ascending=True)  # ascending=True so largest is at top when barh

    colors = []
    for feat in mean_shap.index:
        if feat in filtered_out_s1:
            colors.append('#e74c3c')   # red: removed by correlation filter
        elif feat in zeroed_by_enet:
            colors.append('#e67e22')   # orange: zeroed by ElasticNet
        else:
            colors.append('#3498db')   # blue: retained by all

    bars = ax.barh(range(len(mean_shap)), mean_shap.values, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(mean_shap)))
    ax.set_yticklabels(mean_shap.index, fontsize=9)
    ax.set_xlabel('Mean |SHAP value| (MW)', fontsize=9)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Legend
legend_patches = [
    mpatches.Patch(color='#3498db', label='Retained by all strategies'),
    mpatches.Patch(color='#e74c3c', label='Removed by correlation filter (S1)'),
    mpatches.Patch(color='#e67e22', label='Zeroed by ElasticNet (S4)'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=3,
           fontsize=10, bbox_to_anchor=(0.5, -0.02), frameon=True)

plt.suptitle('Mean |SHAP| on Stage-1 Test Residuals\n(which predictors explain what each strategy missed?)',
             fontsize=13, y=1.01)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(f'{PLOT_DIR}/interpretability_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {PLOT_DIR}/interpretability_shap_bar.png')

In [ ]:
# SHAP beeswarm plots (individual figures — shap.summary_plot creates its own figure)
for name, label in zip(strategy_names, strategy_labels):
    shap.summary_plot(
        shap_values[name], X_shap,
        plot_type='dot', max_display=18, show=False
    )
    plt.title(f'SHAP Beeswarm — {label}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    fname = f'{PLOT_DIR}/interpretability_shap_beeswarm_{name.lower()}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: {fname}')

In [ ]:
# H2 evidence: information loss from correlation filtering
# Measure: for S1 (Filter), what fraction of total |SHAP| is attributable to filtered-out features?
# Compare same measure for S3 (Ridge) — Ridge includes those features, so they should have lower residual SHAP

print('=== H2: Information Loss from Correlation Filtering ===')
print(f'Features removed by S1 correlation filter: {filtered_out_s1}')
print()

info_loss_rows = []
for name in ['Filter', 'Ridge', 'ElasticNet']:
    mean_shap = pd.Series(np.abs(shap_values[name]).mean(axis=0), index=s3_cols)
    total_shap    = mean_shap.sum()
    # SHAP for features removed by S1
    shap_filtered = mean_shap[mean_shap.index.isin(filtered_out_s1)].sum()
    # SHAP for features zeroed by ElasticNet
    shap_enet_z   = mean_shap[mean_shap.index.isin(zeroed_by_enet)].sum()
    pct_filtered  = 100 * shap_filtered / total_shap
    pct_enet_z    = 100 * shap_enet_z   / total_shap

    info_loss_rows.append({
        'Strategy': name,
        'Total |SHAP| (MW)': round(total_shap, 2),
        '|SHAP| for S1-filtered features (MW)': round(shap_filtered, 2),
        '% attributed to S1-filtered': round(pct_filtered, 2),
        '|SHAP| for ElasNet-zeroed (MW)': round(shap_enet_z, 2),
        '% attributed to ElasNet-zeroed': round(pct_enet_z, 2),
        'XGB R2 on residuals': round(xgb_r2[name], 4),
    })
    print(f'{name:<12}: total|SHAP|={total_shap:.1f} MW  |  '
          f'S1-filtered features = {shap_filtered:.1f} MW ({pct_filtered:.1f}%)  |  '
          f'XGB R2 = {xgb_r2[name]:.4f}')

info_loss_df = pd.DataFrame(info_loss_rows)
print()
print('Interpretation:')
filt_pct_filter = info_loss_df.loc[info_loss_df['Strategy']=='Filter', '% attributed to S1-filtered'].values[0]
filt_pct_ridge  = info_loss_df.loc[info_loss_df['Strategy']=='Ridge',  '% attributed to S1-filtered'].values[0]
if filt_pct_filter > filt_pct_ridge:
    print(f'  H2 SUPPORTED: S1-filtered features explain {filt_pct_filter:.1f}% of Filter residuals '
          f'vs {filt_pct_ridge:.1f}% of Ridge residuals.')
    print(f'  This {filt_pct_filter - filt_pct_ridge:.1f}pp gap quantifies the information discarded by correlation filtering.')
else:
    print(f'  H2 NOT SUPPORTED: S1-filtered features explain {filt_pct_filter:.1f}% of Filter residuals '
          f'vs {filt_pct_ridge:.1f}% of Ridge residuals (no clear information loss).')

---
## §3 Hypothesis Testing

### H1 — Regularisation stability vs accuracy tradeoff

> *Ridge and Elastic Net produce more stable coefficients than correlation filtering across cross-validation folds, without sacrificing multi-step forecast accuracy.*

**Evidence framework:**
- Stability: median CV score and mean sign consistency from §1
- Accuracy: MAPE at h=1,7,30,90,180 days from notebook 03
- H1 is supported if Ridge/ElasticNet have lower median CV **and** competitive or better MAPE at ≥ 4 of 6 horizons

In [ ]:
# Load MAPE results from notebook 03
mape_df = pd.read_csv(f'{DATA_DIR}/sarimax_mape_table.csv', index_col='Model')

# Stability summary (from CV analysis)
stability_summary = pd.DataFrame({
    'Strategy': ['Filter (OLS)', 'Ridge', 'ElasticNet'],
    'Median CV': [
        cv_hm['Filter (OLS)'].median(),
        cv_hm['Ridge'].median(),
        cv_hm['ElasticNet'].median(),
    ],
    'Mean Sign Consistency': [
        sign_hm['Filter (OLS)'].mean(),
        sign_hm['Ridge'].mean(),
        sign_hm['ElasticNet'].mean(),
    ],
}).set_index('Strategy').round(3)

print('--- Coefficient Stability ---')
print(stability_summary.to_string())
print()
print('--- Forecast MAPE (%) by Horizon ---')
print(mape_df.to_string())
print()

# Map strategy names to MAPE rows
strategy_mape_map = {
    'Filter (OLS)': 'SARIMAX-1 (Filter)',
    'Ridge':        'SARIMAX-3 (Ridge)',
    'ElasticNet':   'SARIMAX-4 (ElasNet)',
}
horizons = [c for c in mape_df.columns if c.startswith('h=')]

# H1 test: compare Ridge and ElasticNet vs Filter on stability + accuracy
filter_mape  = mape_df.loc['SARIMAX-1 (Filter)', horizons].values.astype(float)
ridge_mape   = mape_df.loc['SARIMAX-3 (Ridge)',  horizons].values.astype(float)
enet_mape    = mape_df.loc['SARIMAX-4 (ElasNet)', horizons].values.astype(float)

ridge_wins   = (ridge_mape  <= filter_mape).sum()
enet_wins    = (enet_mape   <= filter_mape).sum()
n_horizons   = len(horizons)

filter_cv  = cv_hm['Filter (OLS)'].median()
ridge_cv   = cv_hm['Ridge'].median()
enet_cv    = cv_hm['ElasticNet'].median()

print('--- H1 Assessment ---')
print(f'Ridge   : median CV = {ridge_cv:.3f} vs Filter {filter_cv:.3f} '
      f'({"more" if ridge_cv < filter_cv else "less"} stable)  |  '
      f'MAPE wins = {ridge_wins}/{n_horizons} horizons')
print(f'ElasNet : median CV = {enet_cv:.3f} vs Filter {filter_cv:.3f} '
      f'({"more" if enet_cv < filter_cv else "less"} stable)  |  '
      f'MAPE wins = {enet_wins}/{n_horizons} horizons')

h1_ridge = (ridge_cv < filter_cv) and (ridge_wins >= n_horizons // 2)
h1_enet  = (enet_cv  < filter_cv) and (enet_wins  >= n_horizons // 2)
print(f'H1 SUPPORTED for Ridge     : {h1_ridge}')
print(f'H1 SUPPORTED for ElasticNet: {h1_enet}')

In [ ]:
# H1 visualisation: stability–accuracy tradeoff scatter + MAPE horizon comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Stability vs accuracy (at h=1d MAPE)
strategies_plot = ['Filter (OLS)', 'Ridge', 'ElasticNet']
colors_plot     = ['#e74c3c', '#2980b9', '#27ae60']
cv_vals         = [cv_hm[s].median() for s in strategies_plot]
mape_1d         = [
    float(mape_df.loc['SARIMAX-1 (Filter)', 'h=1d']),
    float(mape_df.loc['SARIMAX-3 (Ridge)',  'h=1d']),
    float(mape_df.loc['SARIMAX-4 (ElasNet)', 'h=1d']),
]

for cv_v, mape_v, strat, col in zip(cv_vals, mape_1d, strategies_plot, colors_plot):
    axes[0].scatter(cv_v, mape_v, s=180, color=col, zorder=3, label=strat)
    axes[0].annotate(strat, (cv_v, mape_v), textcoords='offset points',
                     xytext=(8, 4), fontsize=9)
axes[0].set_xlabel('Median CV (lower = more stable)', fontsize=10)
axes[0].set_ylabel('h=1d MAPE (%)', fontsize=10)
axes[0].set_title('Stability vs Short-Horizon Accuracy\n(ideal: lower-left)', fontsize=11, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].annotate('Ideal region', xy=(min(cv_vals)*0.9, min(mape_1d)*0.98),
                  fontsize=8, color='grey', style='italic')

# Right: MAPE across horizons
h_labels = ['h=1d', 'h=3d', 'h=7d', 'h=30d', 'h=90d', 'h=180d']
h_num    = [1, 3, 7, 30, 90, 180]
for strat_name, sarimax_name, col in zip(
    strategies_plot,
    ['SARIMAX-1 (Filter)', 'SARIMAX-3 (Ridge)', 'SARIMAX-4 (ElasNet)'],
    colors_plot
):
    mape_vals = [float(mape_df.loc[sarimax_name, h]) for h in h_labels]
    axes[1].plot(h_num, mape_vals, marker='o', color=col, label=strat_name, linewidth=2)

axes[1].set_xscale('log')
axes[1].set_xticks(h_num)
axes[1].set_xticklabels(h_labels, rotation=20)
axes[1].set_xlabel('Forecast horizon (days, log scale)', fontsize=10)
axes[1].set_ylabel('MAPE (%)', fontsize=10)
axes[1].set_title('Forecast Accuracy Across Horizons', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('H1: Regularisation — Stability vs Accuracy Tradeoff', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/interpretability_h1_stability_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {PLOT_DIR}/interpretability_h1_stability_accuracy.png')

In [ ]:
# H2 visualisation: SHAP importance of filtered-out features across strategies
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: total |SHAP| for S1-filtered features per strategy
strats_h2 = ['Filter', 'Ridge', 'ElasticNet']
colors_h2 = ['#e74c3c', '#2980b9', '#27ae60']
shap_filtered_vals = []
shap_total_vals    = []
for name in strats_h2:
    ms = pd.Series(np.abs(shap_values[name]).mean(axis=0), index=s3_cols)
    shap_filtered_vals.append(ms[ms.index.isin(filtered_out_s1)].sum())
    shap_total_vals.append(ms.sum())

x = np.arange(len(strats_h2))
axes[0].bar(x, shap_total_vals,    color=[c + '44' for c in ['#e74c3c', '#2980b9', '#27ae60']],
            label='Total |SHAP|', edgecolor='grey')
axes[0].bar(x, shap_filtered_vals, color=colors_h2, label=f'|SHAP| for {filtered_out_s1} (filtered by S1)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(strats_h2)
axes[0].set_ylabel('Mean |SHAP value| (MW)')
axes[0].set_title(f'H2: SHAP Attribution to\nFiltered-Out Features ({filtered_out_s1})',
                  fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.2, axis='y')

# Right: percentage of total SHAP explained by filtered-out features
pct_filtered = [100 * s / t for s, t in zip(shap_filtered_vals, shap_total_vals)]
bars = axes[1].bar(x, pct_filtered, color=colors_h2, edgecolor='grey')
for bar, pct in zip(bars, pct_filtered):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{pct:.1f}%', ha='center', fontsize=10, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(strats_h2)
axes[1].set_ylabel('% of total |SHAP| from filtered features')
axes[1].set_title('Proportion of Residual Variance\nExplained by Filtered-Out Features',
                  fontsize=11, fontweight='bold')
axes[1].grid(True, alpha=0.2, axis='y')

plt.suptitle('H2: Information Loss from Correlation Filtering\n'
             '(higher % in Filter residuals = more information discarded)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/interpretability_h2_information_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {PLOT_DIR}/interpretability_h2_information_loss.png')

### H3 — LSTM accuracy gap scaffold

> *LSTM significantly outperforms the best SARIMAX variant at short horizons (h = 1, 3, 7 days) but not at long horizons (h = 30, 90, 180 days).*

**Evidence required (from notebook 05):**
- LSTM MAPE at h = 1, 3, 7, 30, 90, 180 days
- Diebold-Mariano test statistic and p-value comparing best SARIMAX vs LSTM at each horizon
- H3 is supported if DM p < 0.05 for h ≤ 7d (LSTM better) and p ≥ 0.05 for h ≥ 30d (no significant difference)

The cell below sets up the comparison framework and will be completed once `data/lstm_results.csv` is available from notebook 05.

In [ ]:
# H3 scaffold: LSTM vs best SARIMAX comparison
# Load best SARIMAX per horizon (from notebook 03)
best_sarimax = pd.read_csv(f'{DATA_DIR}/sarimax_best_per_horizon.csv')
print('Best SARIMAX per horizon (benchmark for H3):')
print(best_sarimax[['horizon_days', 'model', 'mape_pct', 'nrmse_pct']].to_string(index=False))
print()

lstm_results_path = f'{DATA_DIR}/lstm_results.csv'
if os.path.exists(lstm_results_path):
    lstm_df = pd.read_csv(lstm_results_path)
    print('LSTM results found — computing accuracy gap:')
    comparison = best_sarimax.merge(
        lstm_df[['horizon_days', 'mape_pct']].rename(columns={'mape_pct': 'lstm_mape_pct'}),
        on='horizon_days'
    )
    comparison['gap_pct'] = comparison['mape_pct'] - comparison['lstm_mape_pct']
    comparison['LSTM_better'] = comparison['gap_pct'] > 0
    print(comparison[['horizon_days', 'model', 'mape_pct', 'lstm_mape_pct', 'gap_pct', 'LSTM_better']].to_string(index=False))

    # Diebold-Mariano test (requires prediction-level data from both models)
    print()
    print('Note: Diebold-Mariano test requires per-prediction series from both models.')
    print('Implement DM test in notebook 06_results_tables.ipynb after LSTM is trained.')
else:
    print(f'LSTM results not yet available ({lstm_results_path}).')
    print('Run notebook 05_lstm_benchmark.ipynb first, then re-run this cell.')
    print()
    print('H3 framework: expected outcome based on forecasting literature:')
    for h, expectation in [(1, 'LSTM significantly better (DM p < 0.05)'),
                            (7, 'LSTM marginally better'),
                            (30, 'No significant difference expected'),
                            (180,'SARIMAX may be comparable or better (structural predictors dominate)')]:
        best_row = best_sarimax[best_sarimax['horizon_days'] == h]
        if len(best_row) > 0:
            print(f'  h={h:3d}d: best SARIMAX MAPE={best_row.iloc[0]["mape_pct"]:.2f}%  |  {expectation}')

In [ ]:
# Physical sign consistency analysis — thesis section on interpretability
# Check whether coefficients respect physical priors

print('=== Physical Sign Consistency Check ===')
print('Priors: temp_c(-), nao(-), price_eur_kwh(-), is_holiday(-), is_weekend(-)')
print()

physical_priors = {
    'temp_c':       'negative (higher temp -> lower heating demand)',
    'nao':          'negative (positive NAO -> milder winters -> lower demand)',
    'price_eur_kwh':'negative (higher price -> less consumption, esp. post-2022)',
    'is_holiday':   'negative (holidays -> lower commercial/industrial demand)',
    'is_weekend':   'negative (weekends -> lower demand)',
    'wind_ms':      'ambiguous (wind chill in winter vs mild conditions)',
}

sign_check_rows = []
for feat, prior_desc in physical_priors.items():
    row = {'Feature': feat, 'Physical prior': prior_desc[:40]}
    for strat, mean_c in [('Filter', mean_s1), ('Ridge', mean_s3), ('ElasticNet', mean_s4)]:
        if feat in mean_c.index and not pd.isna(mean_c[feat]) and mean_c[feat] != 0:
            sign_str = f'{mean_c[feat]:+.1f}'
        else:
            sign_str = 'zeroed'
        row[strat] = sign_str
    sign_check_rows.append(row)

sign_check_df = pd.DataFrame(sign_check_rows).set_index('Feature')
print(sign_check_df.to_string())
print()
print('Note: price_eur_kwh sign may be positive in early training period (2009-2020 price pre-shock);')
print('the 2022 demand-elasticity regime shift is a known structural break in this dataset.')

In [ ]:
# Save all outputs

# 1. Coefficient stability table
coef_stability_out = pd.DataFrame({
    'feature':              all_feats,
    'filter_mean_coef':     [mean_s1.get(f, np.nan) for f in all_feats],
    'filter_cv':            [cv_s1.get(f, np.nan)   for f in all_feats],
    'filter_sign_cons':     [sign_s1.get(f, np.nan) for f in all_feats],
    'ridge_mean_coef':      [mean_s3.get(f, np.nan) for f in all_feats],
    'ridge_cv':             [cv_s3.get(f, np.nan)   for f in all_feats],
    'ridge_sign_cons':      [sign_s3.get(f, np.nan) for f in all_feats],
    'elasticnet_mean_coef': [mean_s4.get(f, np.nan) for f in all_feats],
    'elasticnet_cv':        [cv_s4.get(f, np.nan)   for f in all_feats],
    'elasticnet_sign_cons': [sign_s4.get(f, np.nan) for f in all_feats],
})
coef_stability_out.to_csv(f'{DATA_DIR}/interpretability_coef_stability.csv', index=False)
print(f'Saved: {DATA_DIR}/interpretability_coef_stability.csv')

# 2. SHAP summary (mean |SHAP| per feature × strategy)
shap_summary_rows = []
for name in strategy_names:
    ms = pd.Series(np.abs(shap_values[name]).mean(axis=0), index=s3_cols)
    for feat in s3_cols:
        shap_summary_rows.append({
            'strategy': name,
            'feature':  feat,
            'mean_abs_shap': round(ms[feat], 4),
            'removed_by_s1_filter': feat in filtered_out_s1,
            'zeroed_by_enet':       feat in zeroed_by_enet,
        })
shap_df = pd.DataFrame(shap_summary_rows)
shap_df.to_csv(f'{DATA_DIR}/interpretability_shap_summary.csv', index=False)
print(f'Saved: {DATA_DIR}/interpretability_shap_summary.csv')

# 3. H1 summary
h1_out = stability_summary.copy()
h1_out.to_csv(f'{DATA_DIR}/interpretability_h1_summary.csv')
print(f'Saved: {DATA_DIR}/interpretability_h1_summary.csv')

# 4. H2 info loss
info_loss_df.to_csv(f'{DATA_DIR}/interpretability_h2_info_loss.csv', index=False)
print(f'Saved: {DATA_DIR}/interpretability_h2_info_loss.csv')

print()
print('=== Notebook 04 complete ===')
print(f'Outputs: coef_stability.csv, shap_summary.csv, h1_summary.csv, h2_info_loss.csv')
print(f'Plots  : {PLOT_DIR}/interpretability_*.png (5 files)')